# Solving for static equilibrium
This notebook will help you assess in simulation which of the sphere configurations in the problem represent configurations at equilibrium and which. **You do not need to turn in this notebook, and there is no autograded component.** It is just to help you build intuition, show you how to use Drake for problems like this, and check your answers!

## Imports and function definitions

In [1]:
# python libraries
import numpy as np
import pydrake
from pydrake.all import (
    AddMultibodyPlantSceneGraph,
    DiagramBuilder,
    MeshcatVisualizer,
    RigidTransform,
    RotationMatrix,
    Simulator,
    Solve,
    StartMeshcat,
    StaticEquilibriumProblem,
)

from manipulation import running_as_notebook
from manipulation.scenarios import AddShape

In [2]:
# Start the visualizer.
meshcat = StartMeshcat()

INFO:drake:Meshcat listening for connections at http://localhost:7001


## Initialization

In [ ]:
mu = 0.5
r = 0.3
m = 1

builder = DiagramBuilder()
plant, scene_graph = AddMultibodyPlantSceneGraph(builder, time_step=1e-4)
plant.set_name("plant")

world_offset_frame = plant.AddFrame(
    pydrake.multibody.tree.FixedOffsetFrame( # type: ignore
        "world_joint_frame",
        plant.world_frame(),
        RigidTransform(RotationMatrix.MakeXRotation(np.pi / 2), [0, 0, 0]), # type: ignore
    )
)

# Create the sphere bodies
spheres = []
sphere_joints = []
for i in range(3):
    sphere_name = "sphere_{}".format(i)

    color = [0, 0, 0, 1]
    color[i] = 1
    spheres.append(
        AddShape(
            plant,
            pydrake.geometry.Sphere(r), # type: ignore
            name=sphere_name,
            mass=m,
            mu=mu, # type: ignore
            color=color,
        )
    )

    # Set up planar joint
    sphere_joints.append(
        plant.AddJoint(
            pydrake.multibody.tree.PlanarJoint( # type: ignore
                "sphere_{}_joint".format(i),
                world_offset_frame,
                plant.GetFrameByName(sphere_name),
            )
        )
    )

ground = AddShape(plant, pydrake.geometry.Box(10, 10, 2.0), name="ground", mu=mu) # type: ignore
plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("ground"),
    RigidTransform(p=[0, 0, -1.0]), # type: ignore
)

plant.Finalize()

visualizer = MeshcatVisualizer.AddToBuilder(builder, scene_graph, meshcat)

diagram = builder.Build()
context = diagram.CreateDefaultContext()
plant_context = plant.GetMyMutableContextFromRoot(context)

# Using the plant
This is the main of the notebook for you to edit. (The other spot is where the system parameters are defined near the top of the script.) There are three sections:

1. **Initializing your guess for a static equilibrium position**: You can specify the $xyz$ position of each of the sphere. (To answer the question, you'll want to make it match one of the configurations from the problem, but feel free to experiment/try others.)
2. **Computing the static equilibrium position**: The `StaticEquilibriumProblem` class allows us to automatically set up the optimization problem for static equilibrium for a given plant. We use this class to compute an actual equilibrium position.
3. **Simulating the plant.** Given a configuration for the system, simulate how it evolves over time.

## Initializing your guess for a static equilibrium position
Specify the x and z of the center of mass of each of the spheres. (The spheres are fixed in the $xz$ plane, so that's all you have to specify.)

In [54]:
#########
# REPLACE WITH YOUR CODE
guesses = [
    [0, r],  # Red sphere xz
    [2*r, r],  # Green sphere xz
    [r, 4*r],  # Blue sphere xz
]
#########

### Visualizing your guess
Run the following cell to see your guess rendered in meshcat. **This does not check for static equilibrium or run any physics simulation,** but it will let you verify you've set your pose how you intended.

In [55]:
for i, guess in enumerate(guesses):
    sphere_joints[i].set_translation(plant_context, guess)
diagram.ForcedPublish(context)

## Computing the static equilibrium position
This cell computes a static equilibrium postion. If it's close to your original guess, then you initialized the system at equilibrium. If not, your guess is not an equilibrium.

In [56]:
# The StaticEquilibriumProblem needs an "autodiff" version of the diagram/multibody plant to
# use gradient-based optimization.
autodiff_diagram = diagram.ToAutoDiffXd()
autodiff_context = autodiff_diagram.CreateDefaultContext()
autodiff_plant = autodiff_diagram.GetSubsystemByName("plant") # type: ignore
static_equilibrium_problem = StaticEquilibriumProblem(
    autodiff_plant,
    autodiff_plant.GetMyContextFromRoot(autodiff_context),
    set(),
)

initial_guess = np.zeros(plant.num_positions())

for i, guess in enumerate(guesses):
    initial_guess[3 * i] = guess[0]  # x
    initial_guess[3 * i + 1] = guess[1]  # z

static_equilibrium_problem.get_mutable_prog().SetInitialGuess(
    static_equilibrium_problem.q_vars(), initial_guess
)

result = Solve(static_equilibrium_problem.prog())
result.is_success()
q_sol = result.GetSolution(static_equilibrium_problem.q_vars())

for i, guess in enumerate(guesses):
    print(
        "Guess for position of {}:".format(i),
        guess,
        "\tEquilibrium position of sphere {}:".format(i),
        q_sol[3 * i : 3 * i + 2],
    )

for wrench in static_equilibrium_problem.GetContactWrenchSolution(result):
    print(
        f"Spatial force at world position {wrench.p_WCb_W} between {wrench.bodyA_index} and {wrench.bodyB_index}:"
    )
    print(f"  translational: {wrench.F_Cb_W.translational()}")
    print(f"  rotational: {wrench.F_Cb_W.rotational()}")

Guess for position of 0: [0, 0.3] 	Equilibrium position of sphere 0: [-0.29999999  0.30010194]
Guess for position of 1: [0.6, 0.3] 	Equilibrium position of sphere 1: [0.89999999 0.30010194]
Guess for position of 2: [0.3, 1.2] 	Equilibrium position of sphere 2: [0.3 0.3]
Spatial force at world position [5.99999991e-01 1.83759438e-17 3.00101937e-01] between BodyIndex(1) and BodyIndex(2):
  translational: [7.99664826e-06 0.00000000e+00 1.21027896e-16]
  rotational: [0. 0. 0.]
Spatial force at world position [4.32962255e-09 1.83728229e-17 3.00050968e-01] between BodyIndex(1) and BodyIndex(3):
  translational: [1.35881874e-09 0.00000000e+00 7.99800696e-06]
  rotational: [0. 0. 0.]
Spatial force at world position [-2.99999991e-01  1.83759438e-17  0.00000000e+00] between BodyIndex(1) and BodyIndex(4):
  translational: [-7.99800708e-06  0.00000000e+00 -9.81000800e+00]
  rotational: [0. 0. 0.]
Spatial force at world position [5.99999996e-01 1.83728229e-17 3.00050968e-01] between BodyIndex(2) an

### Visualizing the solution configuration
This doesn't yet run the dynamics for the system (so the objects won't move), but it *will* update their poses to match the results of `StaticEquilibriumProblem`.

In [57]:
plant.SetPositions(plant_context, q_sol)
diagram.ForcedPublish(context)

## Simulating the solution

You may see simulations of the static equilibrium that result in the spheres moving.  Why is that?

Keep in mind that
- A static equilibrium might not be a *stable* equilibrium.  States close to the equilibrium might diverge.
- The optimization solver satisfies the equations only up to a numerical tolerance.

In [58]:
simulator = Simulator(diagram)
plant.SetPositions(plant.GetMyContextFromRoot(simulator.get_mutable_context()), q_sol)
if running_as_notebook:
    simulator.set_target_realtime_rate(1.0)
    simulator.AdvanceTo(5.0)
else:
    simulator.AdvanceTo(0.1);

KeyboardInterrupt: 